# State und Mocks – Übersicht

Dieses Notebook **importiert** `state.py` und `mocks.py` aus dem Repo-Root, es ersetzt sie nicht.
Änderungen am State gehören in `state.py` (per Pull Request, Owner: Melvin).

Inhalt:
1. Wer schreibt welche Felder? (Vertrag zwischen den Nodes)
2. Was das Modell beim Structured Output sieht
3. Mocks im Überblick und Plausibilitätsprüfung
4. `mock_bis`: die Eingabe für jeden Node
5. Graph-Skelett mit Dummy-Nodes (Verdrahtung, Schleifen, Reducer)

In [2]:
import json
import sys
from pathlib import Path

# Repo-Root in den Pfad, damit state.py und mocks.py importierbar sind
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from IPython.display import Markdown, display

from state import (
    FELDER_PRO_NODE, NODE_REIHENFOLGE, MAX_VERSUCHE,
    AnalyseErgebnis, BriefState, Bestaetigung, GuardrailErgebnis, KalenderErgebnis,
)
from mocks import MOCKS, MAHNUNG_INJECTION_ERWARTET, mock_bis

## 1. Wer schreibt welche Felder?

Jeder Node darf alles Vorherige lesen, aber nur seine eigenen Felder schreiben.
Das ist die Tabelle «liest/schreibt» aus SW5, Folie 6 (Schritt 3: State entwerfen).

In [3]:
ZUSTAENDIG = {
    "eingabe": "UI (Florian)", "ocr": "Nayan", "analyse": "Nayan",
    "zustelldatum_erfragen": "Melvin", "fristen": "Vendon", "erklaerung": "Vendon",
    "antwort": "Melvin", "guardrail": "Florian", "bestaetigung": "Melvin", "kalender": "Melvin",
}
zeilen = ["| Node | zuständig | schreibt |", "|---|---|---|"]
for node in NODE_REIHENFOLGE:
    felder = ", ".join(f"`{f}`" for f in FELDER_PRO_NODE[node])
    zeilen.append(f"| {node} | {ZUSTAENDIG[node]} | {felder} |")
display(Markdown("\n".join(zeilen)))

| Node | zuständig | schreibt |
|---|---|---|
| eingabe | UI (Florian) | `dokument_pfad`, `nutzersprache` |
| ocr | Nayan | `textquelle`, `rohtext`, `seiten`, `text_lesbar`, `text_konfidenz`, `versuche`, `abbruch_grund` |
| analyse | Nayan | `brieftyp`, `brieftyp_konfidenz`, `brieftyp_vermutung`, `briefsprache`, `antwortsprache`, `injection_verdacht`, `extraktion`, `zustelldatum`, `zustelldatum_quelle` |
| zustelldatum_erfragen | Melvin | `zustelldatum`, `zustelldatum_quelle` |
| fristen | Vendon | `fristen` |
| erklaerung | Vendon | `erklaerung` |
| antwort | Melvin | `antwortentwurf` |
| guardrail | Florian | `guardrail`, `versuche`, `abbruch_grund` |
| bestaetigung | Melvin | `bestaetigung`, `fristen`, `antwortentwurf`, `abbruch_grund` |
| kalender | Melvin | `kalender` |

## 2. Was das Modell beim Structured Output sieht

`llm.with_structured_output(AnalyseErgebnis)` schickt dieses JSON-Schema ans Modell
(SW5, Folie 17). Die `description`-Texte sind damit Teil des Prompts. Wenn die Extraktion
schlecht ist, lohnt sich oft zuerst ein Blick hierher.

In [4]:
schema = AnalyseErgebnis.model_json_schema()
print(f"Grösse: {len(json.dumps(schema))} Zeichen\n")
for name, feld in schema["$defs"]["Extraktion"]["properties"].items():
    print(f"{name:28} {feld.get('description', '')[:90]}")

Grösse: 6828 Zeichen

absender                     Amt oder Firma, die den Brief verschickt
briefdatum                   Datum, das oben auf dem Brief steht
zustelldatum_im_brief        Nur wenn ein Zustelldatum ausgefüllt auf dem Brief steht. Ein leeres Feld 'Zugestellt am: 
referenzen                   Alle Nummern im Brief: Betreibungs-, Steuer-, Kunden-, Policen-, Debitoren-, Rechnungsnumm
periode                      Steuerperiode oder Gültigkeitsjahr, z. B. '2025' oder '2027'
hauptbetrag_chf              Der Betrag, um den es im Brief hauptsächlich geht: Total der Betreibung (ohne Zins), offen
zahlungsrichtung             'zu_zahlen' bei Forderung, 'rueckerstattung' bei Guthaben (z. B. Steuer), 'information' we
posten                       Einzelbeträge aus der Tabelle des Briefs
frist_im_brief               Nur ein konkretes Datum, das im Brief als Frist steht (z. B. Mahnung, Krankenkasse). Relat
glaeubiger                   Wer das Geld fordert (bei Betreibung nicht das Betreib

In [5]:
# Vollständiges Schema bei Bedarf:
# print(json.dumps(schema, indent=2, ensure_ascii=False))

## 3. Mocks im Überblick

Je ein Brief pro Person, aufgebaut aus den synthetischen Beispielbriefen, plus die
Injection-Variante für den Guardrail.

Die Summe der Posten muss dem Hauptbetrag entsprechen. Diese Prüfung ist deterministisch
und eignet sich auch im echten Analyse-Node als Plausibilitätscheck.

In [6]:
zeilen = ["| Mock | Typ | Brief → Nutzer | Hauptbetrag | Richtung | Fristen | Posten ok |",
          "|---|---|---|---|---|---|---|"]
for name, m in MOCKS.items():
    e = m["extraktion"]
    ohne_periode = [p.betrag_chf for p in e.posten if p.periode is None]
    ok = "–" if not ohne_periode else ("✓" if round(sum(ohne_periode), 2) == e.hauptbetrag_chf else "✗")
    fristen = ", ".join(f"{f.art} {f.datum:%d.%m.}" for f in m.get("fristen", []))
    zeilen.append(f"| {name} | {m['brieftyp']} | {m['briefsprache']} → {m['nutzersprache']} "
                  f"| {e.hauptbetrag_chf:,.2f}".replace(",", "'") + f" | {e.zahlungsrichtung} | {fristen} | {ok} |")
display(Markdown("\n".join(zeilen)))

| Mock | Typ | Brief → Nutzer | Hauptbetrag | Richtung | Fristen | Posten ok |
|---|---|---|---|---|---|---|
| zahlungsbefehl | zahlungsbefehl | de → de | 1'928.90 | zu_zahlen | rechtsvorschlag 06.11., zahlung 16.11. | ✓ |
| steuerveranlagung | steuerveranlagung | fr → de | 656.90 | zu_zahlen | einsprache 25.11. | ✓ |
| krankenkasse_praemie | krankenkasse_praemie | it → it | 1'068.10 | information | kuendigung 30.11., franchise_wechsel 30.11. | – |
| mahnung | mahnung | de → de | 506.35 | zu_zahlen | zahlung 06.11. | ✓ |
| mahnung_injection | mahnung | de → de | 506.35 | zu_zahlen | zahlung 06.11. | ✓ |

## 4. `mock_bis`: die Eingabe für jeden Node

`mock_bis(brief, nach_node)` liefert den State, wie er **nach** dem genannten Node aussieht.
Für den eigenen Node nehmt ihr also den Node davor, z. B. der Erklär-Agent
`mock_bis("zahlungsbefehl", "fristen")`.

In [7]:
state = mock_bis("zahlungsbefehl", "fristen")
print("Felder:", sorted(state))
print()
for f in state["fristen"]:
    print(f.model_dump())

Felder: ['abbruch_grund', 'antwortsprache', 'briefsprache', 'brieftyp', 'brieftyp_konfidenz', 'brieftyp_vermutung', 'dokument_pfad', 'extraktion', 'fristen', 'injection_verdacht', 'nutzersprache', 'rohtext', 'seiten', 'text_konfidenz', 'text_lesbar', 'textquelle', 'versuche', 'zustelldatum', 'zustelldatum_quelle']

{'art': 'rechtsvorschlag', 'datum': datetime.date(2026, 11, 6), 'basis': 'zustellung', 'ausgangsdatum': datetime.date(2026, 10, 27), 'rechtsgrundlage': 'SchKG Art. 74', 'hinweise': [], 'erinnerung': datetime.date(2026, 11, 2), 'bestaetigt': False}
{'art': 'zahlung', 'datum': datetime.date(2026, 11, 16), 'basis': 'zustellung', 'ausgangsdatum': datetime.date(2026, 10, 27), 'rechtsgrundlage': 'SchKG Art. 69', 'hinweise': [], 'erinnerung': datetime.date(2026, 11, 11), 'bestaetigt': False}


Welche Felder sind nach welchem Node vorhanden? (Beispiel Zahlungsbefehl)

In [8]:
alle_felder = [f for n in NODE_REIHENFOLGE for f in FELDER_PRO_NODE[n]]
alle_felder = list(dict.fromkeys(alle_felder))
kopf = "| Feld | " + " | ".join(n[:6] for n in NODE_REIHENFOLGE) + " |"
zeilen = [kopf, "|---" * (len(NODE_REIHENFOLGE) + 1) + "|"]
staende = {n: mock_bis("zahlungsbefehl", n) for n in NODE_REIHENFOLGE}
for feld in alle_felder:
    zeilen.append(f"| `{feld}` | " + " | ".join("●" if feld in staende[n] else "" for n in NODE_REIHENFOLGE) + " |")
display(Markdown("\n".join(zeilen)))

| Feld | eingab | ocr | analys | zustel | friste | erklae | antwor | guardr | bestae | kalend |
|---|---|---|---|---|---|---|---|---|---|---|
| `dokument_pfad` | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `nutzersprache` | ● | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `textquelle` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `rohtext` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `seiten` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `text_lesbar` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `text_konfidenz` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `versuche` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `abbruch_grund` |  | ● | ● | ● | ● | ● | ● | ● | ● | ● |
| `brieftyp` |  |  | ● | ● | ● | ● | ● | ● | ● | ● |
| `brieftyp_konfidenz` |  |  | ● | ● | ● | ● | ● | ● | ● | ● |
| `brieftyp_vermutung` |  |  | ● | ● | ● | ● | ● | ● | ● | ● |
| `briefsprache` |  |  | ● | ● | ● | ● | ● | ● | ● | ● |
| `antwortsprache` |  |  | ● | ● | ● | ● | ● | ● | ● | ● |
| `injection_verdacht` |  |  | ● | ● | ● | ● | ● | ● | ● | ● |
| `extraktion` |  |  | ● | ● | ● | ● | ● | ● | ● | ● |
| `zustelldatum` |  |  | ● | ● | ● | ● | ● | ● | ● | ● |
| `zustelldatum_quelle` |  |  | ● | ● | ● | ● | ● | ● | ● | ● |
| `fristen` |  |  |  |  | ● | ● | ● | ● | ● | ● |
| `erklaerung` |  |  |  |  |  | ● | ● | ● | ● | ● |
| `antwortentwurf` |  |  |  |  |  |  | ● | ● | ● | ● |
| `guardrail` |  |  |  |  |  |  |  | ● | ● | ● |
| `bestaetigung` |  |  |  |  |  |  |  |  | ● | ● |
| `kalender` |  |  |  |  |  |  |  |  |  | ● |

## 5. Graph-Skelett mit Dummy-Nodes

Der vollständige Ablauf aus `Entscheidungen.md`, aber jeder Node gibt nur Werte aus dem
Mock zurück. So lässt sich die Verdrahtung testen, bevor ein Agent fertig ist. Wer seinen
Node fertig hat, ersetzt den Dummy durch die echte Funktion.

Zu sehen sind:
- **Conditional Edges** (SW5, Folie 8): unlesbar, unbekannter Typ, fehlendes Zustelldatum,
  Guardrail-Schleife
- **Der Reducer** `zaehler_addieren`: OCR- und Guardrail-Zähler überschreiben sich nicht

Human-in-the-loop (Interrupt) kommt in Woche 8 dazu; hier bestätigt der Dummy automatisch.

In [9]:
from langgraph.graph import END, START, StateGraph

def dummy(node: str, mock: BriefState):
    """Gibt die Felder zurück, die dieser Node laut Vertrag schreibt, aus dem Mock."""
    def ausfuehren(state: BriefState) -> dict:
        update = {k: mock[k] for k in FELDER_PRO_NODE[node]
                  if k in mock and k not in ("versuche", "abbruch_grund")}
        if node == "ocr":
            update["versuche"] = {"ocr": 1}
        if node == "guardrail":
            # Dummy-Regel: bei Injection-Verdacht durchfallen lassen
            bestanden = not state.get("injection_verdacht", False)
            update["guardrail"] = GuardrailErgebnis(
                bestanden=bestanden, zurueck_zu=None if bestanden else "erklaerung")
            update["versuche"] = {"guardrail": 1}
        if node == "bestaetigung":
            update = {"bestaetigung": Bestaetigung(bestaetigt=True)}
        if node == "kalender":
            update = {"kalender": KalenderErgebnis(kanal="ics", ics_pfad="ausgabe/test.ics")}
        return update
    return ausfuehren


# --- Routing (bedingte Kanten) ------------------------------------------------

def nach_ocr(state):
    if state.get("text_lesbar"):
        return "analyse"
    if state.get("versuche", {}).get("ocr", 0) < MAX_VERSUCHE["ocr"]:
        return "ocr"            # in der App: Nutzer um neues Foto bitten
    return END

def nach_analyse(state):
    if state["brieftyp"] == "unbekannt":
        return "erklaerung"     # nur allgemeine Hilfe, keine Frist
    braucht_zustellung = state["brieftyp"] in ("zahlungsbefehl", "steuerveranlagung")
    if braucht_zustellung and state.get("zustelldatum") is None:
        return "zustelldatum_erfragen"
    return "fristen"

def nach_guardrail(state):
    g = state["guardrail"]
    if g.bestanden:
        return "bestaetigung"
    if state["versuche"].get("guardrail", 0) >= MAX_VERSUCHE["guardrail"]:
        return END              # abbruch_grund = "guardrail_limit"
    return "antwort" if g.zurueck_zu == "antwort" else "erklaerung"

def nach_bestaetigung(state):
    return "kalender" if state["bestaetigung"].bestaetigt and state.get("fristen") else END


def baue_graph(mock: BriefState):
    b = StateGraph(BriefState)
    for node in NODE_REIHENFOLGE[1:]:          # "eingabe" ist der Start-State
        b.add_node(node, dummy(node, mock))
    b.add_edge(START, "ocr")
    b.add_conditional_edges("ocr", nach_ocr, ["analyse", "ocr", END])
    b.add_conditional_edges("analyse", nach_analyse,
                            ["erklaerung", "zustelldatum_erfragen", "fristen"])
    b.add_edge("zustelldatum_erfragen", "fristen")
    b.add_edge("fristen", "erklaerung")
    b.add_edge("erklaerung", "antwort")
    b.add_edge("antwort", "guardrail")
    b.add_conditional_edges("guardrail", nach_guardrail,
                            ["bestaetigung", "erklaerung", "antwort", END])
    b.add_conditional_edges("bestaetigung", nach_bestaetigung, ["kalender", END])
    b.add_edge("kalender", END)
    return b.compile()

Die Struktur als Mermaid-Diagramm (z. B. in [mermaid.live](https://mermaid.live) einfügen):

In [10]:
graph = baue_graph(MOCKS["zahlungsbefehl"])
print(graph.get_graph().draw_mermaid())
# Als Bild (braucht Internet): display(Image(graph.get_graph().draw_mermaid_png()))

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	ocr(ocr)
	analyse(analyse)
	zustelldatum_erfragen(zustelldatum_erfragen)
	fristen(fristen)
	erklaerung(erklaerung)
	antwort(antwort)
	guardrail(guardrail)
	bestaetigung(bestaetigung)
	kalender(kalender)
	__end__([<p>__end__</p>]):::last
	__start__ --> ocr;
	analyse -.-> erklaerung;
	analyse -.-> fristen;
	analyse -.-> zustelldatum_erfragen;
	antwort --> guardrail;
	bestaetigung -.-> __end__;
	bestaetigung -.-> kalender;
	erklaerung --> antwort;
	fristen --> erklaerung;
	guardrail -.-> __end__;
	guardrail -.-> antwort;
	guardrail -.-> bestaetigung;
	guardrail -.-> erklaerung;
	ocr -.-> __end__;
	ocr -.-> analyse;
	zustelldatum_erfragen --> fristen;
	kalender --> __end__;
	ocr -.-> ocr;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### Durchlauf mit jedem Mock

Für jeden Brief: welche Nodes durchlaufen werden und wie die Zähler am Ende stehen.
Erwartung: Der Zahlungsbefehl geht über `fristen` (Zustelldatum ist bekannt), die
Injection-Mahnung läuft zweimal durch den Guardrail und bricht dann ab.

In [11]:
for name, mock in MOCKS.items():
    graph = baue_graph(mock)
    start = mock_bis(name, "eingabe")
    pfad = [node for schritt in graph.stream(start, stream_mode="updates") for node in schritt]
    ende = graph.invoke(start)
    print(f"{name}\n  Pfad:     {' → '.join(pfad)}\n  Versuche: {ende['versuche']}\n")

zahlungsbefehl
  Pfad:     ocr → analyse → fristen → erklaerung → antwort → guardrail → bestaetigung → kalender
  Versuche: {'ocr': 1, 'guardrail': 1}

steuerveranlagung
  Pfad:     ocr → analyse → fristen → erklaerung → antwort → guardrail → bestaetigung → kalender
  Versuche: {'ocr': 1, 'guardrail': 1}

krankenkasse_praemie
  Pfad:     ocr → analyse → fristen → erklaerung → antwort → guardrail → bestaetigung → kalender
  Versuche: {'ocr': 1, 'guardrail': 1}

mahnung
  Pfad:     ocr → analyse → fristen → erklaerung → antwort → guardrail → bestaetigung → kalender
  Versuche: {'ocr': 1, 'guardrail': 1}

mahnung_injection
  Pfad:     ocr → analyse → fristen → erklaerung → antwort → guardrail → erklaerung → antwort → guardrail
  Versuche: {'ocr': 1, 'guardrail': 2}



### Variante: Zustelldatum fehlt

Ohne Zustelldatum muss der Graph beim Zahlungsbefehl über `zustelldatum_erfragen` gehen.

In [12]:
import copy
ohne_zustellung = copy.deepcopy(MOCKS["zahlungsbefehl"])
ohne_zustellung["zustelldatum"] = None
graph = baue_graph(ohne_zustellung)
pfad = [n for s in graph.stream(mock_bis("zahlungsbefehl", "eingabe"), stream_mode="updates") for n in s]
print("Zahlungsbefehl ohne Zustelldatum:", " → ".join(pfad))
print()
print("Routing direkt getestet:")
print("  Zahlungsbefehl ohne Datum →", nach_analyse({"brieftyp": "zahlungsbefehl", "zustelldatum": None}))
print("  Krankenkasse ohne Datum   →", nach_analyse({"brieftyp": "krankenkasse_praemie", "zustelldatum": None}))
print("  Unbekannter Brief         →", nach_analyse({"brieftyp": "unbekannt", "zustelldatum": None}))

Zahlungsbefehl ohne Zustelldatum: ocr → analyse → zustelldatum_erfragen → fristen → erklaerung → antwort → guardrail → bestaetigung → kalender

Routing direkt getestet:
  Zahlungsbefehl ohne Datum → zustelldatum_erfragen
  Krankenkasse ohne Datum   → fristen
  Unbekannter Brief         → erklaerung


## Nächste Schritte pro Person

Jede Person legt ein eigenes Notebook für ihren Node an und startet mit:

```python
from mocks import mock_bis
state = mock_bis("<brief>", "<node davor>")
update = mein_node(state)
```

Danach den Dummy in `baue_graph` durch die echte Funktion ersetzen und den Durchlauf oben
erneut ausführen.